# [fake-useragent](https://github.com/fake-useragent/fake-useragent)

Библиотека `fake-useragent` - это стандарт для подмены `User-Agent` в Python, позволяющий скриптам выдавать себя за реальные браузеры и снижать риск блокировки при парсинге.

## 1. Введение: Зачем подменять User-Agent?

**User-Agent (UA)** - это строка, которую браузер или скрипт отправляет серверу при каждом запросе, идентифицируя себя (тип приложения, операционная система, версия браузера). Когда мы используем `requests` или `aiohttp` без настройки заголовков, сервер видит что-то вроде `python-requests/2.23.0`, что мгновенно идентифицирует трафик как бот.

Многие сайты блокируют такие запросы. Библиотека `fake-useragent` решает эту проблему, предоставляя простой интерфейс для генерации огромного количества актуальных и правдоподобных `User-Agent` строк реальных браузеров (Chrome, Firefox, Safari и др.) .
**Важно:** Подмена `User-Agent` - это лишь первый уровень маскировки. Современные сайты могут применять и другие методы защиты: ограничение частоты запросов, анализ других заголовков (Cookie, Accept-Language и др.), проверка выполнения JavaScript, CAPTCHAs и блокировки по IP-адресу. Поэтому кроме смены `User-Agent` часто нужны дополнительные меры (задержки между запросами, прокси, эмуляция браузера и т.п.).


## 2. Установка

Установка производится стандартными средствами Python.

```bash
pip install fake-useragent
```

Или для обновления уже установленной версии:
```bash
pip install --upgrade fake-useragent
```

Убедиться, что установка прошла успешно, можно выполнив в Python-консоли:
```python
import fake_useragent
print(fake_useragent.__version__)
```



## 3. Основы использования: Первые шаги

С библиотекой очень легко начать работу. Весь функционал сосредоточен в классе `UserAgent`.

### 3.1. Простейший пример

Создание экземпляра класса и вызов свойства `.random` для получения случайной строки `User-Agent`.

In [ ]:
from fake_useragent import UserAgent

# Инициализация объекта UserAgent
ua = UserAgent()

# Получение случайного User-Agent
random_ua = ua.random
print(random_ua)
# Пример вывода: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36

Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36 Edg/135.0.0.0


### 3.2. Интеграция с Requests

Это самый распространенный сценарий использования - подстановка случайного `User-Agent` в каждый HTTP-запрос.  
Чтобы сервер видел не `python-requests`, а строку реального браузера.

In [10]:
import requests
from fake_useragent import UserAgent

ua = UserAgent()
url = 'https://httpbin.org/headers'  # Сервис для эхо-проверки заголовков

# Генерируем случайный User-Agent для этого запроса
headers = {
    'User-Agent': ua.random
}

response = requests.get(url, headers=headers)
print(response.json().get('headers').get('User-Agent'))

Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.6 Safari/605.1.15


## 4. Расширенные стратегии ротации

Просто использовать `ua.random` для каждого запроса - уже что-то, но для масштабируемых проектов существуют более тонкие настройки.

### 4.1. Выбор конкретного браузера

Если нам нужно имитировать только один тип браузера (например, для тестирования вёрстки), необходимо использовать соответствующие конкретным браузерам атрибуты.


In [11]:
from fake_useragent import UserAgent

ua = UserAgent()

print(ua.chrome)   # Случайная версия Chrome
print(ua.google)   # Альтернативный способ для Chrome
print(ua.firefox)  # Firefox
print(ua.ff)       # Сокращение для Firefox
print(ua.safari)   # Safari
print(ua.edge)     # Microsoft Edge
print(ua.opera)    # Opera


Mozilla/5.0 (Linux; Android 10; K) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36
Mozilla/5.0 (iPhone; CPU iPhone OS 18_3_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) GSA/363.0.743255906 Mobile/15E148 Safari/604.1
Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:137.0) Gecko/20100101 Firefox/137.0
Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:137.0) Gecko/20100101 Firefox/137.0
Mozilla/5.0 (iPhone; CPU iPhone OS 18_3_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.3.1 Mobile/15E148 Safari/604.1
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36 Edg/135.0.0.0
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36 OPR/117.0.0.0 (Edition std-2)


### 4.2. Тонкая фильтрация (браузеры, ОС, платформы)

Начиная с версии 1.2.0, `fake-useragent` позволяет гибко фильтровать генерируемые строки при инициализации объекта. Это мощный инструмент для создания "цифрового отпечатка".

Для начала, можно генерировать `User-Agent` только для определенных браузеров:
```python
# Список поддерживаемых браузеров для параметра browsers в fake_useragent
browsers = ["Google", "Chrome", "Firefox", "Edge", "Opera", "Safari", "Android", 
"Yandex Browser", "Samsung Internet", "Opera Mobile", "Mobile Safari", "Firefox Mobile",
 "Firefox iOS", "Chrome Mobile", "Chrome Mobile iOS", "Mobile Safari UI/WKWebView", 
"Edge Mobile", "DuckDuckGo Mobile", "MiuiBrowser", "Whale", "Twitter", "Facebook", "Amazon Silk"]
```

In [22]:
# Фильтрация по браузерам

# Генерировать только User-Agent для Chrome и Edge
ua = UserAgent(browsers=['Edge', 'Chrome'])
print(ua.random)

Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36


По умолчанию `fake-useragent` использует все распространённые ОС:
```python
# Список поддерживаемых операционных систем для параметра os в fake_useragent
os = ["Windows", "Linux", "Ubuntu", "Chrome OS", "Mac OS X", "Android", "iOS"]
```

⚠️ Важно: Начиная с версии 2.0.0 имена ОС чувствительны к регистру. Если напиcать os='linux' (с маленькой буквы), то это не сработает и может привести к исключению `FakeUserAgentError`.

In [21]:
# Фильтрация по операционной системе

# Генерировать только User-Agent для Linux
ua = UserAgent(os='Linux')
print(ua.random)

# Для мобильных устройств
ua_mobile = UserAgent(platforms=['mobile'])
print(ua_mobile.random)

Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36
Mozilla/5.0 (Linux; Android 10; K) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Mobile Safari/537.36


In [20]:
# Комбинирование фильтров

# Десктопный Firefox на Windows
ua = UserAgent(browsers=['Firefox'], platforms=['desktop'], os=['Windows'])
print(ua.random)

Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:137.0) Gecko/20100101 Firefox/137.0


⚠️ Примечание: Названия браузеров и ОС чувствительны к регистру. Полный список доступных значений можно найти в [официальной документации](https://github.com/fake-useragent/fake-useragent).

Также возможны фильтрации по параметрам:

* `platform` - по типу устройства:
    - `desktop`
    - `mobile`
    - `tablet`
* `min_version` - по минимальной версии браузера

In [32]:
from fake_useragent import UserAgent

ua = UserAgent(
    os='iOS',
    browsers=['Mobile Safari', 'Chrome'],
    platforms='mobile',
    min_version=18.0
)

print(ua.random)

Mozilla/5.0 (iPhone; CPU iPhone OS 18_3_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.3.1 Mobile/15E148 Safari/604.1


### 4.3. Работа со словарем User-Agent (данные, а не только строка)

Начиная с версии 1.3.0, библиотека позволяет получать не просто строку, а целый словарь с данными о доле использования, версии браузера и ОС. Это может быть полезно для аналитики или A/B тестирования.

In [26]:
from fake_useragent import UserAgent

ua = UserAgent()

# Получить случайный объект User-Agent
ua_obj = ua.getRandom
print(ua_obj)
# Пример вывода:
# {
#   'useragent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 ...',
#   'percent': 0.8,
#   'type': 'desktop'
#   'browser': 'Chrome',
#   'browser_version': '135.0.0.0',
#   'os': 'Mac OS X'
# }

# Получить конкретный браузер как объект
chrome_obj = ua.getChrome
print(chrome_obj.get('os'))

{'useragent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36', 'percent': 0.010466391697894735, 'type': 'desktop', 'device_brand': None, 'browser': 'Chrome', 'browser_version': '135.0.0.0', 'browser_version_major_minor': 135.0, 'os': 'Windows', 'os_version': '10', 'platform': 'Win32'}
iOS


## 5. Продвинутые техники и обработка ошибок

### 5.1. Обработка сбоев: параметр `fallback`

В крайне редких случаях (проблемы с сетью, кэшем) библиотека может не смочь сгенерировать строку. Чтобы скрипт не падал с ошибкой, можно использовать `fallback` - запасной вариант.

In [28]:
from fake_useragent import UserAgent

ua = UserAgent(fallback='Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)')
# Если что-то пойдет не так, .random вернет эту строку
print(ua.random)

Mozilla/5.0 (iPhone; CPU iPhone OS 18_4 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.4 Mobile/15E148 Safari/604.1


### 5.2. Кэширование для ускорения

По умолчанию библиотека скачивает актуальную базу `User-Agent`'ов при первом обращении и кэширует её. Это разумное поведение, но мы можем управлять кэшем.

Если мы сталкиваемся с устаревшими данными или хотим принудительно обновить кэш, необходимо использовать параметр `cache_path` для указания своего файла кэша или просто удалить старый. Местоположение кэша по умолчанию зависит от ОС, но чаще всего это временная директория. Мы также можем отключить кэш (не рекомендуется, так как это замедлит работу и создаст лишнюю нагрузку на серверы данных).


In [ ]:
# Пример использования своего пути для кэша (концептуально)
# ua = UserAgent(cache_path='/tmp/my_cache.json')

### 5.3. Имитация полного набора заголовков

Как отмечают эксперты, отправка одного лишь `User-Agent` недостаточна. Реальный браузер отправляет десятки заголовков: `Accept`, `Accept-Language`, `Sec-CH-UA`, `Referer` и другие. Библиотека `fake-useragent` генерирует только UA-строку. Формирование полного набора заголовков - это наша задача.

Для этого необходимо создать "базовый" набор заголовков реального браузера и подставлять в него сгенерированный `ua.random`.

In [29]:
import requests
from fake_useragent import UserAgent

ua = UserAgent()

def get_browser_headers():
    """Возвращает словарь заголовков, имитирующих реальный браузер."""
    return {
        'User-Agent': ua.random,
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Accept-Encoding': 'gzip, deflate, br',
        'DNT': '1', # Do Not Track
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Sec-Fetch-Dest': 'document',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-Site': 'none',
        'Sec-Fetch-User': '?1',
        'Cache-Control': 'max-age=0',
    }

response = requests.get('https://httpbin.org/headers', headers=get_browser_headers())
print(response.json())

{'headers': {'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8', 'Accept-Encoding': 'gzip, deflate, br', 'Accept-Language': 'en-US,en;q=0.5', 'Cache-Control': 'max-age=0', 'Dnt': '1', 'Host': 'httpbin.org', 'Sec-Fetch-Dest': 'document', 'Sec-Fetch-Mode': 'navigate', 'Sec-Fetch-Site': 'none', 'Sec-Fetch-User': '?1', 'Upgrade-Insecure-Requests': '1', 'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36', 'X-Amzn-Trace-Id': 'Root=1-69a7f6f5-19b7b9770eafe3b169e9dc26'}}


## 6. Практические примеры использования

### 6.1. Ротация в цикле (Scrapy-style)

Пример для циклического парсинга.

In [31]:
import requests
from fake_useragent import UserAgent
import time

ua = UserAgent()
urls = ['https://httpbin.org/headers', 'https://httpbin.org/headers', 'https://httpbin.org/headers']

for url in urls:
    headers = {'User-Agent': ua.random}
    try:
        response = requests.get(url, headers=headers, timeout=5)
        print(f"Обработан {url} с UA: {headers['User-Agent'][:50]}...")
        # Обязательная задержка между запросами
        time.sleep(2)
    except requests.RequestException as e:
        print(f"Ошибка при запросе {url}: {e}")

Обработан https://httpbin.org/headers с UA: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWeb...
Обработан https://httpbin.org/headers с UA: Mozilla/5.0 (X11; CrOS x86_64 14541.0.0) AppleWebK...
Обработан https://httpbin.org/headers с UA: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWeb...


### 6.2. Использование с библиотекой `aiohttp` (асинхронно)

`fake-useragent` не является асинхронной библиотекой, но её вызовы (генерация строки) синхронны и быстры, поэтому их можно безопасно использовать в асинхронном коде.

In [ ]:
import aiohttp
import asyncio
from fake_useragent import UserAgent

ua = UserAgent()

async def fetch(session, url):
    headers = {'User-Agent': ua.random}
    async with session.get(url, headers=headers) as response:
        return await response.text()

async def main():
    async with aiohttp.ClientSession() as session:
        html = await fetch(session, 'https://httpbin.org/headers')
        print(html)

asyncio.run(main())

## 7. Возможные ошибки и проблемы

* **CAPTCHA и JS-защита**: Подмена `User-Agent` не обходит капчу и защиты типа `Cloudflare`. Если сайт при подозрительной активности выдает капчу или требует выполнения `JavaScript`, простой HTTP-запрос с любым `User-Agent` её не пройдёт.
* **Блокировка по IP-адресу**: При слишком частом обращении к сайту, он может начать блокировать наш IP. Смена `User-Agent` не спасёт, так как сервер распознает повторяющиеся запросы с одного адреса. Признаки блокировки: резкие ошибки 403/503, пустые ответы или редирект на страницу с "подозрительной активностью".
* **Другие заголовки и поведение**: Некоторые продвинутые сайты проверяют не только `User-Agent`, но и другие заголовки на соответствие. Например, если скрипт выдаёт себя за Chrome, но не отправляет заголовки `Accept-Language`, `Accept` или не загружает связанные ресурсы, сайт может заподозрить, что вы бот. По возможности стоит отправлять набор заголовков, типичный для выбранного `User-Agent` (язык, типы контента и пр.). Также полезно обрабатывать `cookies` и сессии, как это делает браузер.
* **Ограничения самой библиотеки**: `fake-useragent` генерирует строки агентов случайно, и иногда может вернуть довольно экзотичные или устаревшие варианты. Это не ошибка, но нужно учитывать: если сайт рассчитан только на современные браузеры, а наш скрипт представится, скажем, Internet Explorer 9, то сайт может отдать упрощённую версию страницы или показать уведомление о неподдерживаемом браузере.

## 8. Заключение и лучшие практики

Библиотека `fake-useragent` - это незаменимый инструмент для любого Python-разработчика, занимающегося сбором данных. Она проста в использовании, но для достижения максимальной эффективности необходимо придерживаться следующих правил:

1.  **Всегда обновлять библиотеку:** База User-Agent'ов устаревает, новые версии браузеров выходят часто. Необходимо держать `fake-useragent` в актуальном состоянии.
2.  **Использовать фильтры:** Если мы парсим мобильную версию сайта, генерировать нужно мобильные User-Agent'ы. Это сделает трафик более правдоподобным.
3.  **Комбинировать с другими заголовками:** Не останавливаться на User-Agent. Использовать полный набор HTTP-заголовков.
4.  **Добавлять задержки и прокси:** Ротация User-Agent'ов - это лишь один слой защиты. Также нужно чередовать его с ротацией IP-адресов через прокси и соблюдать вежливые задержки между запросами.
5.  **Всегда иметь fallback:** Защитить свой код от непредвиденных сбоев можно с помощью параметра `fallback`.